In [ ]:
import numpy as np
import open3d as o3d
from numpy.lib.format import read_magic, _read_array_header
import os


points_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/env_0_points.npy"
colors_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/env_0_colors.npy"   # varsa kullan
use_colors = True   # renk dosyan yoksa False yap


def load_npy_safely(path):
    with open(path, "rb") as f:
        version = read_magic(f)
        shape, fortran_order, dtype = _read_array_header(f, version)
        data_start = f.tell()

    print(f"\nLOADING: {path}")
    print("version:", version)
    print("header shape:", shape)
    print("fortran_order:", fortran_order)
    print("dtype:", dtype)
    print("data starts at byte:", data_start)
    print("file size:", os.path.getsize(path))

    raw = np.fromfile(path, dtype=dtype, offset=data_start)
    print("raw.size:", raw.size)

    if fortran_order:
        arr = raw.reshape(shape, order="F")
    else:
        arr = raw.reshape(shape)

    print("recovered shape:", arr.shape)
    print(arr[:5])
    return arr


def align_normals_with_centroid(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)

    centroid = np.mean(pts, axis=0)
    for i in range(len(normals)):
        to_point = pts[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        _, idx, _ = kdtree.search_knn_vector_3d(pts[i], 10)
        avg_normal = np.mean(normals[idx], axis=0)
        norm = np.linalg.norm(avg_normal)

        if norm > 1e-12:
            avg_normal = avg_normal / norm
            if np.dot(normals[i], avg_normal) < 0:
                normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud



# load target points directly
points = load_npy_safely(points_path)
points = np.asarray(points, dtype=float)

assert points.ndim == 2 and points.shape[1] >= 3, "points must be Nx3 or Nx>=3"
points = points[:, :3]
assert len(points) > 0, "No points loaded"

print("\nLoaded target points shape:", points.shape)
print(points[:5])

if use_colors:
    colors = load_npy_safely(colors_path)
    colors = np.asarray(colors, dtype=float)

    assert colors.ndim == 2 and colors.shape[1] >= 3, "colors must be Nx3 or Nx>=3"
    colors = colors[:, :3]

    if colors.max() > 1.0:
        colors = colors / 255.0

    assert len(points) == len(colors), f"Point/color count mismatch: {len(points)} vs {len(colors)}"

    print("\nLoaded target colors shape:", colors.shape)
    print(colors[:5])
else:
    colors = None



# visualize raw target object

target_pcd = o3d.geometry.PointCloud()
target_pcd.points = o3d.utility.Vector3dVector(points)

if colors is not None:
    target_pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([target_pcd], window_name="Raw Target Object")



# downsample

pcd_tmp = o3d.geometry.PointCloud()
pcd_tmp.points = o3d.utility.Vector3dVector(points)

if colors is not None:
    pcd_tmp.colors = o3d.utility.Vector3dVector(colors)

pcd_tmp = pcd_tmp.voxel_down_sample(voxel_size=0.005)

print("\nDownsampled point count:", len(pcd_tmp.points))
if colors is not None:
    print("Downsampled color count:", len(pcd_tmp.colors))



# estimate normals

pcd_tmp.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

pcd_tmp = align_normals_with_centroid(pcd_tmp)
pcd_tmp = reorient_normals_locally(pcd_tmp)

normals = np.asarray(pcd_tmp.normals)

print("Number of processed points:", len(pcd_tmp.points))
print("Number of normals:", len(normals))

o3d.visualization.draw_geometries(
    [pcd_tmp],
    window_name="Target Object with Normals",
    point_show_normal=True
)


LOADING: /mnt/c/Users/user/Documents/CS549/grasp_gen/env_0_points.npy
version: (1, 0)
header shape: (15159877, 3)
fortran_order: True
dtype: float64
data starts at byte: 128
file size: 363837176
raw.size: 45479631
recovered shape: (15159877, 3)
[[-1.03559387e+00  1.03559387e+00  2.33650208e-05]
 [-1.03451512e+00  1.03559387e+00  2.33650208e-05]
 [-1.03343638e+00  1.03559387e+00  2.33650208e-05]
 [-1.03235764e+00  1.03559387e+00  2.33650208e-05]
 [-1.03127889e+00  1.03559387e+00  2.33650208e-05]]

Loaded target points shape: (15159877, 3)
[[-1.03559387e+00  1.03559387e+00  2.33650208e-05]
 [-1.03451512e+00  1.03559387e+00  2.33650208e-05]
 [-1.03343638e+00  1.03559387e+00  2.33650208e-05]
 [-1.03235764e+00  1.03559387e+00  2.33650208e-05]
 [-1.03127889e+00  1.03559387e+00  2.33650208e-05]]

LOADING: /mnt/c/Users/user/Documents/CS549/grasp_gen/env_0_colors.npy
version: (1, 0)
header shape: (15159877, 3)
fortran_order: False
dtype: float32
data starts at byte: 128
file size: 181918652
ra

In [ ]:
def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

min_distance = 0.01
max_distance = 0.12
small_angle_threshold = 10  # Max angle 
large_angle_threshold = 160  # Min angle 

best_grasps = []
compute_grasp_points = True
cropped_pcl = np.asarray(pcd_tmp.points)
if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []

    centroid = np.mean(points, axis=0)

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)
            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)
                if 160 <= angle <= 180:
                    # Compute the epipolar line (direction vector)
                    epiline = p2 - p1
                    epiline /= np.linalg.norm(epiline)

                    # Check normal directions relative to the epipolar line
                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                  
                    if projection_n1 < 0 and projection_n2 > 0:
                        # Calculate the angles between the epipolar line and each normal
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)
                        if min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold:
                            # Calculate closeness to the centroid
                            dist_to_centroid = np.linalg.norm((p1 + p2) / 2 - centroid)
                            grasp_candidates.append((p1, p2, n1, n2, angle, distance, angle_n1_epiline, angle_n2_epiline, dist_to_centroid))
    if grasp_candidates:
        # Select the best grasp considering  closeness to the centroid
        best_grasp = min(grasp_candidates, key=lambda x: x[8] * 1e1)  # Minimize closeness to centroid
        best_grasps.append(best_grasp)
        print(f"Best grasp candidate:", best_grasp)
    else:
        print(f"No suitable grasp candidates found")
        grasp_candidates.append(None)


Best grasp candidate: (array([ 0.1572043 , -0.66076897,  0.66523116]), array([ 0.18400947, -0.56442726,  0.67988118]), array([-0.19618394, -0.98056062, -0.00356705]), array([0.23196606, 0.96125194, 0.14895119]), 171.34224080451952, 0.10106861881861143, 170.8404817175926, 1.9732671040926872, 0.00044549447412523153)


In [ ]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    # grasp_candidates_np = np.array(grasp_candidates, dtype=object)

    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
    # np.savez("grasp_candidates.npz", grasp_candidates=grasp_candidates_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']
    # grasp_candidates = loaded_data['grasp_candidates']

In [ ]:
for i, best_grasp in enumerate(best_grasps):


    p1 = best_grasp[0]
    p2 = best_grasp[1]
    n1 = best_grasp[2]
    n2 = best_grasp[3]

    # diplay selected points and their normals in open3d
    point_cloud = o3d.geometry.PointCloud()

    point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
    point_cloud.colors = o3d.utility.Vector3dVector([[255, 0, 0], [0, 255, 0]])

    normals = np.array([n1, n2])
    point_cloud.normals = o3d.utility.Vector3dVector(normals)

    grasp_visualizations = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere1.translate(p1)
    sphere1.paint_uniform_color([1, 0, 0])  

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere2.translate(p2)
    sphere2.paint_uniform_color([0, 1, 0]) 

    # Add spheres to the visualization
    grasp_visualizations.append(sphere1)
    grasp_visualizations.append(sphere2)

    # Create lines to represent normals at each grasp point
    normal_line1 = o3d.geometry.LineSet()
    normal_line2 = o3d.geometry.LineSet()

    # Normal vectors n1 and n2 at points p1 and p2 respectively

    line_points1 = [p1, p1 + 0.05 * n1]  # Line representing normal direction for p1
    line_points2 = [p2, p2 + 0.05 * n2]  # Line representing normal direction for p2
   
    normal_line1.points = o3d.utility.Vector3dVector(line_points1)
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p1 to p1 + normal
    normal_line1.colors = o3d.utility.Vector3dVector([[1, 0, 0]])  # Color red

    normal_line2.points = o3d.utility.Vector3dVector(line_points2)
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p2 to p2 + normal
    normal_line2.colors = o3d.utility.Vector3dVector([[0, 1, 0]])  # Color green

    # Add normal lines to the visualization
    grasp_visualizations.append(normal_line1)
    grasp_visualizations.append(normal_line2)

    o3d.visualization.draw_geometries([point_cloud] + grasp_visualizations)

